# Step 10: Fine-tune DINOv2-Large + UperNet (Full Mapping, 1,328 classes)

Thin orchestration notebook for Databricks. All reusable logic lives in
`histological_image_analysis.training` (installed as a wheel on the cluster).

**Full mapping:** Every structure in the Allen Brain ontology gets its own class
(1,327 structures + background = 1,328 classes). This is the finest granularity
for identifying individual brain structures in tissue sections.

**Memory:** Logits are ~2.7 GB per GPU at batch=4 (fp16). Fits comfortably on L40S 48 GB
(~12% utilization). Run on multi-GPU cluster (8x L40S) for speed.

In [ ]:
# Cell 0 — Install project wheel from DBFS
# The wheel is uploaded by `make deploy-wheel` and contains all reusable
# training pipeline code (ontology, slicer, dataset, training modules).

%pip install /dbfs/FileStore/wheels/histological_image_analysis-0.1.0-py3-none-any.whl
dbutils.library.restartPython()

In [ ]:
# Cell 1 — Configuration
# All environment-specific paths and hyperparameters live here.

import os
import mlflow

# ---------- MLflow setup ----------
os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "true"

EXPERIMENT_NAME = "/Users/noel.nosse@grainger.com/histology-brain-segmentation"
try:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
except Exception:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
mlflow.set_experiment(experiment_id=experiment_id)
print(f"MLflow experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")

# ---------- Databricks paths ----------
WORKSPACE_BASE = "/Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology"
ONTOLOGY_PATH = f"{WORKSPACE_BASE}/ontology/structure_graph_1.json"
ANNOTATION_10_PATH = f"{WORKSPACE_BASE}/ccfv3/annotation_10.nrrd"
# ara_nissl_10.nrrd exceeds workspace limit — lives on DBFS
NISSL_10_PATH = "/dbfs/FileStore/allen_brain_data/ccfv3/ara_nissl_10.nrrd"

# ---------- JFrog / HuggingFace model ----------
HF_ENDPOINT = os.environ.get(
    "HF_ENDPOINT",
    "https://graingerreadonly.jfrog.io/artifactory/api/huggingfaceml/huggingfaceml-remote",
)
MODEL_REPO_ID = "facebook/dinov2-large"
MODEL_CACHE_DIR = "/tmp/dinov2-large"

# ---------- Mapping ----------
MAPPING_TYPE = "full"

# ---------- Training hyperparameters ----------
HYPERPARAMS = {
    "mapping_type": MAPPING_TYPE,
    "crop_size": 518,  # DINOv2 native resolution
    "batch_size": 4,  # Conservative for ~1,328 classes (2.7 GB logits per GPU)
    "learning_rate": 1e-4,
    "num_epochs": 50,
    "freeze_backbone": True,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "split_strategy": "interleaved",
    "model": MODEL_REPO_ID,
    "dataset": "CCFv3 ara_nissl_10",
}

# Unpack for convenience (NUM_LABELS computed dynamically in Cell 3)
CROP_SIZE = HYPERPARAMS["crop_size"]
BATCH_SIZE = HYPERPARAMS["batch_size"]
LEARNING_RATE = HYPERPARAMS["learning_rate"]
NUM_EPOCHS = HYPERPARAMS["num_epochs"]
FREEZE_BACKBONE = HYPERPARAMS["freeze_backbone"]
SPLIT_STRATEGY = HYPERPARAMS["split_strategy"]

# ---------- Output ----------
OUTPUT_DIR = f"/tmp/checkpoints/{MAPPING_TYPE}"
FINAL_MODEL_DIR = f"/dbfs/FileStore/allen_brain_data/models/{MAPPING_TYPE}"

print(f"Ontology:       {ONTOLOGY_PATH}")
print(f"Nissl 10µm:     {NISSL_10_PATH}")
print(f"HF endpoint:    {HF_ENDPOINT}")
print(f"Mapping:        {MAPPING_TYPE}")
print(f"Batch size:     {BATCH_SIZE} (reduced for memory)")
print(f"Split strategy: {SPLIT_STRATEGY}")
print(f"Checkpoints:    {OUTPUT_DIR}")
print(f"Final model:    {FINAL_MODEL_DIR}")

In [ ]:
# Cell 2 — Download model weights from JFrog Artifactory mirror
#
# Uses retry pattern per LESSONS_LEARNED.md — Artifactory can drop connections.
# etag_timeout=86400 avoids unnecessary HEAD requests.

import time
from huggingface_hub import snapshot_download

print(f"Downloading {MODEL_REPO_ID} from Artifactory...")
for attempt in range(1, 4):
    try:
        model_path = snapshot_download(
            repo_id=MODEL_REPO_ID,
            endpoint=HF_ENDPOINT,
            etag_timeout=86400,
            cache_dir=MODEL_CACHE_DIR,
        )
        break
    except Exception as e:
        if attempt < 3:
            wait = 2 ** attempt
            print(f"  Attempt {attempt} failed ({e.__class__.__name__}), retrying in {wait}s...")
            time.sleep(wait)
        else:
            raise

print(f"Model downloaded to: {model_path}")

In [ ]:
# Cell 3 — Build training and validation datasets
#
# Pipeline: OntologyMapper → CCFv3Slicer → BrainSegmentationDataset
#
# Uses full mapping (every structure → unique class) and INTERLEAVED split.

from collections import Counter
from histological_image_analysis.ontology import OntologyMapper
from histological_image_analysis.ccfv3_slicer import CCFv3Slicer
from histological_image_analysis.dataset import BrainSegmentationDataset

# 1. Load ontology and build full mapping
mapper = OntologyMapper(ONTOLOGY_PATH)
mapping = mapper.build_full_mapping()
NUM_LABELS = mapper.get_num_labels(mapping)
class_names = mapper.get_class_names(mapping)
HYPERPARAMS["num_labels"] = NUM_LABELS
print(f"Full mapping: {NUM_LABELS} classes (including background)")

# 2. Load CCFv3 volumes (10µm Nissl + annotations)
slicer = CCFv3Slicer(
    image_path=NISSL_10_PATH,
    annotation_path=ANNOTATION_10_PATH,
    ontology_mapper=mapper,
)
slicer.load_volumes()
print(f"Loaded {slicer.num_slices} coronal slices")

# 3. Interleaved train/val/test split
splits = slicer.get_split_indices(
    train_frac=0.8, val_frac=0.1, split_strategy=SPLIT_STRATEGY,
)
print(f"Train: {len(splits['train'])} | Val: {len(splits['val'])} | Test: {len(splits['test'])}")

# 4. Create datasets
train_ds = BrainSegmentationDataset(
    slicer=slicer, split="train", mapping=mapping,
    crop_size=CROP_SIZE, augment=True, split_strategy=SPLIT_STRATEGY,
)
val_ds = BrainSegmentationDataset(
    slicer=slicer, split="val", mapping=mapping,
    crop_size=CROP_SIZE, augment=False, split_strategy=SPLIT_STRATEGY,
)
print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

# Sanity check: inspect one sample
sample = train_ds[0]
print(f"pixel_values: {sample['pixel_values'].shape}, labels: {sample['labels'].shape}")
print(f"Unique classes in sample: {len(set(sample['labels'].numpy().ravel()))}")

In [ ]:
# Cell 4 — Create model
#
# Frozen backbone, head-only training.
# Forward-pass sanity check included.

import torch
from histological_image_analysis.training import create_model

model = create_model(
    num_labels=NUM_LABELS,
    freeze_backbone=FREEZE_BACKBONE,
    pretrained_backbone_path=model_path,
)

# Sanity check — single forward pass
model.eval()
with torch.no_grad():
    dummy = torch.randn(1, 3, CROP_SIZE, CROP_SIZE)
    if torch.cuda.is_available():
        model = model.cuda()
        dummy = dummy.cuda()
    out = model(pixel_values=dummy)
    print(f"Logits shape: {out.logits.shape}")
    print(f"Expected: (1, {NUM_LABELS}, {CROP_SIZE}, {CROP_SIZE})")
    logits_mb = out.logits.element_size() * out.logits.nelement() / 1e6
    print(f"Logits memory (single sample): {logits_mb:.1f} MB")
    print(f"Logits memory (batch={BATCH_SIZE}): {logits_mb * BATCH_SIZE:.1f} MB")
    assert out.logits.shape == (1, NUM_LABELS, CROP_SIZE, CROP_SIZE)
    model = model.cpu()  # Free GPU for training
print("Forward pass OK")

In [ ]:
# Cell 5 — Train
#
# Single MLflow run spans training + eval + export (per LESSONS_LEARNED.md).
# HF Trainer's MLflowCallback logs per-epoch metrics automatically.

from datetime import datetime
from histological_image_analysis.training import create_trainer, get_training_args

mlflow.start_run(run_name=f"{MAPPING_TYPE}-{NUM_LABELS}class-{datetime.now().strftime('%Y%m%d-%H%M')}")
mlflow.log_params(HYPERPARAMS)

training_args = get_training_args(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    fp16=torch.cuda.is_available(),
    report_to="mlflow",
)

trainer = create_trainer(
    model=model,
    training_args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    num_labels=NUM_LABELS,
)

trainer.train()

In [ ]:
# Cell 6 — Evaluate + Visualize
#
# Logs to the same active MLflow run opened in Cell 5.
# Uses get_class_names() for per-class IoU reporting.

import matplotlib.pyplot as plt
import numpy as np

# Run evaluation
eval_results = trainer.evaluate()
print("Evaluation results:")
for k, v in sorted(eval_results.items()):
    if isinstance(v, float) and not k.startswith("eval_iou_class_"):
        print(f"  {k}: {v:.4f}")

# Log final eval metrics explicitly
mlflow.log_metrics({
    "final_mean_iou": eval_results.get("eval_mean_iou", 0.0),
    "final_overall_accuracy": eval_results.get("eval_overall_accuracy", 0.0),
})

# Per-class IoU summary — show top and bottom classes (too many for full listing)
class_ious = {}
for cls in range(NUM_LABELS):
    iou_key = f"eval_iou_class_{cls}"
    iou = eval_results.get(iou_key, float('nan'))
    if not np.isnan(iou):
        class_ious[cls] = iou

sorted_ious = sorted(class_ious.items(), key=lambda x: x[1], reverse=True)
print(f"\nTop 10 classes by IoU:")
for cls, iou in sorted_ious[:10]:
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    print(f"  {cls:4d} ({name:40s}): {iou:.4f}")

print(f"\nBottom 10 classes by IoU:")
for cls, iou in sorted_ious[-10:]:
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    print(f"  {cls:4d} ({name:40s}): {iou:.4f}")

print(f"\nClasses with non-NaN IoU: {len(class_ious)} / {NUM_LABELS}")

# Visualize predictions vs ground truth on a few val samples
model.eval()
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for i in range(3):
    sample = val_ds[i * (len(val_ds) // 3)]
    pixel_values = sample["pixel_values"].unsqueeze(0)
    labels = sample["labels"].numpy()

    with torch.no_grad():
        if torch.cuda.is_available():
            pixel_values = pixel_values.cuda()
        logits = model(pixel_values=pixel_values).logits
        preds = logits.argmax(dim=1).squeeze().cpu().numpy()

    # Denormalize image for display
    img = sample["pixel_values"].permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    axes[i, 0].imshow(img)
    axes[i, 0].set_title("Input")
    axes[i, 1].imshow(labels, cmap="nipy_spectral")
    axes[i, 1].set_title("Ground Truth")
    axes[i, 2].imshow(preds, cmap="nipy_spectral")
    axes[i, 2].set_title("Prediction")

for ax in axes.ravel():
    ax.axis("off")

plt.suptitle(f"{MAPPING_TYPE} {NUM_LABELS}-class | mIoU={eval_results.get('eval_mean_iou', 0):.3f}")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 — Save final model + close MLflow run

import os

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
trainer.save_model(FINAL_MODEL_DIR)
print(f"Model saved to: {FINAL_MODEL_DIR}")

# Close the MLflow run opened in Cell 5.
# Per LESSONS_LEARNED.md: HF Trainer's MLflowCallback can leave runs open.
mlflow.end_run()
print("MLflow run closed.")